In [1]:
import sys
from pathlib import Path

# add src directory to python path
sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from logistic_regression import pre_process_tools

#### Phase 01 : Import data

In [2]:
pre_processed = pd.read_parquet('../src/data/pre_processed/features.parquet')

In [3]:
elo_prob_index = pd.DataFrame()

# elo_prob_index['index'] = 1.0 / (1.0 + 10 ** (-pre_processed['Elo_diff'] / 400.0))
elo_prob_index['index'] = pre_processed['Elo_diff']
elo_prob_index['winner'] = pre_processed['Winner']

In [4]:
elo_prob_index.head(5)

,index,winner
0,0.0,0
1,0.0,0
2,0.0,1
3,0.0,1
4,0.0,0


#### Phase 02 : Split train/test

In [6]:
y_name = 'winner'
X_train, X_test, y_train, y_test = pre_process_tools.split_data(elo_prob_index, y_name, features=['index'])

In [7]:
print('X train shape:', X_train.shape)
print('X test  shape:', X_test .shape)
print('y train shape:', y_train.shape)
print('y test  shape:', y_test .shape)

X train shape: (54705, 1)
X test  shape: (13677, 1)
y train shape: (54705,)
y test  shape: (13677,)


In [8]:
X_train

array([[  0.        ],
       [  0.        ],
       [  0.        ],
       ...,
       [ 61.41829692],
       [199.39976479],
       [122.48300787]], shape=(54705, 1))

#### Phase 03 : Train

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

base_model = LogisticRegression(
    penalty='elasticnet',
    solver='saga',          # only solver supporting elasticnet
    fit_intercept=False,    # important
    class_weight=None,      # data is already balanced by symmetrization
    max_iter=2000,          # seems segs need lots of iter
    tol=1e-4,
)

param_grid = {
    'C': np.logspace(-4, 2, 13),
    'l1_ratio': [0.0, 0.15, 0.3, 0.5, 0.7, 0.85, 1.0],
}

grid = GridSearchCV(
    base_model,
    param_grid,
    cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_log_loss',
    n_jobs=-1,
    verbose=1,
    refit=True,
)

In [10]:
grid.fit(X_train, y_train)

print("Best params :", grid.best_params_)
print("CV LogLoss  :", -grid.best_score_)

best = grid.best_estimator_

Fitting 5 folds for each of 91 candidates, totalling 455 fits
Best params : {'C': np.float64(0.03162277660168379), 'l1_ratio': 0.3}
CV LogLoss  : 0.6009199809374065


In [11]:
w = pd.Series(best.coef_[0], index=['index'])
print(w)

index    0.004507
dtype: float64


#### Phase 06 : Evaluation

In [12]:
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss, accuracy_score

# Probabilities, not hard classes
proba = best.predict_proba(X_test)[:, 1]

print("Test LogLoss :", log_loss(y_test, proba))
print("Test AUC     :", roc_auc_score(y_test, proba))
print("Test Brier   :", brier_score_loss(y_test, proba))
print("Test Acc     :", accuracy_score(y_test, proba > 0.5))

Test LogLoss : 0.6242274090717461
Test AUC     : 0.705694282755668
Test Brier   : 0.21774751798151257
Test Acc     : 0.642904145645975
